# XGBoost Prediction Plots

This notebook loads the saved XGBoost pipeline artifact, evaluates it on the test split, and saves two plots:

1. Actual vs. Predicted scatter plot (color-coded by price band).
2. Prediction accuracy by price band (actual vs. predicted median prices).

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt


def ensure_project_root() -> Path:
    """Put the repo root on sys.path whether the notebook starts in repo root or notebooks/."""
    cwd = Path.cwd().resolve()
    for root in (cwd, *cwd.parents):
        if (root / "pricing_lab" / "__init__.py").is_file():
            root_str = str(root)
            if root_str not in sys.path:
                sys.path.insert(0, root_str)
            return root
    raise RuntimeError("Could not find project root containing pricing_lab.")


PROJECT_ROOT = ensure_project_root()

from pricing_lab.data import load_train_test
from pricing_lab.diagnostics import build_residual_frame, summarize_segment, top_price_band_warning
from pricing_lab.metrics import compute_dollar_metrics

print("Project root:", PROJECT_ROOT)


In [ ]:
MODEL_PATH = PROJECT_ROOT / "artifacts/xgboost.joblib"
PLOTS_DIR = PROJECT_ROOT / "artifacts/plots"
SCATTER_PLOT_PATH = PLOTS_DIR / "xgboost_actual_vs_predicted_by_band.png"
MEAN_BAND_BAR_PLOT_PATH = PLOTS_DIR / "xgboost_prediction_accuracy_by_band_mean.png"
P90_BAND_BAR_PLOT_PATH = PLOTS_DIR / "xgboost_prediction_accuracy_by_band_p90.png"
MISPRICED_OUTPUT_PATH = PROJECT_ROOT / "artifacts/xgboost_top_mispriced_examples.csv"
BAND_LABELS = ["Low", "Mid", "High"]
BAND_COLORS = {
    "Low": "#2ca02c",
    "Mid": "#1f77b4",
    "High": "#d62728",
}

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing model artifact at {MODEL_PATH}. Run training with --model-output-dir artifacts first."
    )

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
data = load_train_test()
xgboost_pipeline = joblib.load(MODEL_PATH)
predicted_log_prices = np.asarray(xgboost_pipeline.predict(data.X_test), dtype=np.float64)

metrics = compute_dollar_metrics(data.y_test.to_numpy(dtype=np.float64), predicted_log_prices)
evaluation_frame = build_residual_frame(
    data.X_test,
    data.y_test.to_numpy(dtype=np.float64),
    predicted_log_prices,
)
evaluation_frame["price_band"] = pd.qcut(
    evaluation_frame["actual_price"],
    q=len(BAND_LABELS),
    labels=BAND_LABELS,
)
evaluation_frame["pricing_gap"] = evaluation_frame["predicted_price"] - evaluation_frame["actual_price"]
evaluation_frame["absolute_gap"] = evaluation_frame["pricing_gap"].abs()
evaluation_frame["percent_gap"] = evaluation_frame["pricing_gap"] / evaluation_frame["actual_price"].clip(lower=1.0) * 100.0

if not np.all(np.isfinite(evaluation_frame[["actual_price", "predicted_price", "pricing_gap"]].to_numpy())):
    raise ValueError("Evaluation frame contains non-finite price or gap values.")
if set(evaluation_frame["price_band"].cat.categories) != set(BAND_LABELS):
    raise ValueError("Actual price bands did not resolve to the expected Low/Mid/High labels.")

print("XGBoost dollar metrics:", {key: round(value, 4) for key, value in metrics.items()})
display(evaluation_frame.head())


In [ ]:
sns.set_theme(style="whitegrid")
fig, axis = plt.subplots(figsize=(9, 7))

maximum_axis_value = max(
    float(evaluation_frame["actual_price"].max()),
    float(evaluation_frame["predicted_price"].max()),
)
axis.plot(
    [0, maximum_axis_value],
    [0, maximum_axis_value],
    linestyle="--",
    linewidth=2,
    color="black",
    label="Ideal prediction",
)
sns.scatterplot(
    data=evaluation_frame,
    x="actual_price",
    y="predicted_price",
    hue="price_band",
    hue_order=BAND_LABELS,
    palette=BAND_COLORS,
    alpha=0.65,
    s=36,
    ax=axis,
)
axis.set_xlim(0, maximum_axis_value * 1.02)
axis.set_ylim(0, maximum_axis_value * 1.02)
axis.set_title("Actual vs. Predicted Price by Actual Price Band")
axis.set_xlabel("Actual price (USD)")
axis.set_ylabel("Predicted price (USD)")
axis.legend(title="Actual price band")
fig.tight_layout()
fig.savefig(SCATTER_PLOT_PATH, dpi=180)
plt.show()
print(f"Saved scatter plot to {SCATTER_PLOT_PATH}")


## Top under-priced and over-priced listing examples

This deliverable surfaces the largest pricing gaps on the test split:
- **Under-priced candidates**: model predicts much higher than actual listed price.
- **Over-priced candidates**: model predicts much lower than actual listed price.

The table is exported to CSV for reporting and review.

In [ ]:
TOP_MISPRICED_ROWS = 20

mispriced_frame = evaluation_frame.copy()

under_priced = mispriced_frame.nlargest(TOP_MISPRICED_ROWS, "pricing_gap").copy()
under_priced["pricing_direction"] = "under_priced_candidate"

over_priced = mispriced_frame.nsmallest(TOP_MISPRICED_ROWS, "pricing_gap").copy()
over_priced["pricing_direction"] = "over_priced_candidate"

top_mispriced_examples = (
    pd.concat([under_priced, over_priced], ignore_index=True)
    .loc[:, [
        "pricing_direction",
        "neighbourhood_group",
        "neighbourhood",
        "room_type",
        "price_band",
        "actual_price",
        "predicted_price",
        "pricing_gap",
        "absolute_gap",
        "percent_gap",
        "predicted_to_actual_ratio",
        "log_residual",
    ]]
    .sort_values(["pricing_direction", "absolute_gap"], ascending=[True, False])
    .reset_index(drop=True)
)

if len(top_mispriced_examples) != 2 * TOP_MISPRICED_ROWS:
    raise ValueError("Unexpected number of top mispriced rows.")

top_mispriced_examples.to_csv(MISPRICED_OUTPUT_PATH, index=False)
print(f"Saved top mispriced listing examples to {MISPRICED_OUTPUT_PATH}")

display(top_mispriced_examples)


In [ ]:
band_summary = summarize_segment(evaluation_frame, "price_band", min_rows=1).reindex(BAND_LABELS)
print(top_price_band_warning(band_summary))

mean_plot_frame = (
    band_summary[["actual_median", "predicted_median"]]
    .rename(columns={"actual_median": "Actual", "predicted_median": "Predicted"})
    .reset_index(names="Price band")
    .melt(id_vars="Price band", var_name="Series", value_name="Price")
)

rmse_plot_frame = (
    band_summary[["mae", "rmse", "log_rmse", "prediction_ratio", "mape"]]
    .reset_index(names="Price band")
)

p90_summary = (
    evaluation_frame.groupby("price_band", observed=True)
    .agg(
        actual_p90=("actual_price", lambda values: values.quantile(0.9)),
        predicted_p90=("predicted_price", lambda values: values.quantile(0.9)),
    )
    .reindex(BAND_LABELS)
)
p90_plot_frame = (
    p90_summary[["actual_p90", "predicted_p90"]]
    .rename(columns={"actual_p90": "Actual", "predicted_p90": "Predicted"})
    .reset_index(names="Price band")
    .melt(id_vars="Price band", var_name="Series", value_name="Price")
)

sns.set_theme(style="whitegrid", context="talk")
palette = {"Actual": "#4F46E5", "Predicted": "#14B8A6"}

fig_mean, axis_mean = plt.subplots(figsize=(9, 6))
mean_bar_plot = sns.barplot(
    data=mean_plot_frame,
    x="Price band",
    y="Price",
    hue="Series",
    order=BAND_LABELS,
    palette=palette,
    ax=axis_mean,
)
for container in mean_bar_plot.containers:
    axis_mean.bar_label(container, fmt="$%.0f", padding=3, fontsize=10)
axis_mean.set_title("Prediction Accuracy by Actual Price Band: Median", pad=12, fontweight="semibold")
axis_mean.set_xlabel("Actual Price Band")
axis_mean.set_ylabel("Median Price (USD)")
axis_mean.legend(title="", frameon=False, loc="upper left")
axis_mean.spines["top"].set_visible(False)
axis_mean.spines["right"].set_visible(False)
axis_mean.grid(axis="y", linestyle="--", alpha=0.3)
fig_mean.tight_layout()
fig_mean.savefig(MEAN_BAND_BAR_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved median price-band bar plot to {MEAN_BAND_BAR_PLOT_PATH}")

fig_p90, axis_p90 = plt.subplots(figsize=(9, 6))
p90_bar_plot = sns.barplot(
    data=p90_plot_frame,
    x="Price band",
    y="Price",
    hue="Series",
    order=BAND_LABELS,
    palette=palette,
    ax=axis_p90,
)
for container in p90_bar_plot.containers:
    axis_p90.bar_label(container, fmt="$%.0f", padding=3, fontsize=10)
axis_p90.set_title("Prediction Accuracy by Actual Price Band: P90", pad=12, fontweight="semibold")
axis_p90.set_xlabel("Actual Price Band")
axis_p90.set_ylabel("P90 Price (USD)")
axis_p90.legend(title="", frameon=False, loc="upper left")
axis_p90.spines["top"].set_visible(False)
axis_p90.spines["right"].set_visible(False)
axis_p90.grid(axis="y", linestyle="--", alpha=0.3)
fig_p90.tight_layout()
fig_p90.savefig(P90_BAND_BAR_PLOT_PATH, dpi=220, bbox_inches="tight")
plt.show()
print(f"Saved p90 price-band bar plot to {P90_BAND_BAR_PLOT_PATH}")

display(rmse_plot_frame)
band_summary
